In [14]:
import re

def parse_expression(expr_str):
    expr_str = expr_str.replace(' ', '')
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', expr_str)
    if match:
        pred = match.group(1)
        args_str = match.group(2)
        args = []
        depth = 0
        current = []
        for char in args_str:
            if char == '(':
                depth += 1
            elif char == ')':
                depth -= 1
            if char == ',' and depth == 0:
                args.append(''.join(current))
                current = []
            else:
                current.append(char)
        if current:
            args.append(''.join(current))
        return [pred] + [parse_expression(arg) for arg in args]
    return expr_str

def is_variable(expr):
    return isinstance(expr, str) and expr[0].islower() if expr else False

def apply_substitution(expr, substitution):
    if is_variable(expr):
        if expr in substitution:
            return apply_substitution(substitution[expr], substitution)
        return expr
    if isinstance(expr, list):
        return [expr[0]] + [apply_substitution(arg, substitution) for arg in expr[1:]]
    return expr

def occurs_in(var, expr, substitution):
    expr_sub = apply_substitution(expr, substitution)
    if var == expr_sub:
        return True
    if isinstance(expr_sub, list):
        return any(occurs_in(var, arg, substitution) for arg in expr_sub[1:])
    return False

def combine(s1, s2):
    combined = {}
    for var, val in s1.items():
        combined[var] = apply_substitution(val, s2)
    for var, val in s2.items():
        if var not in combined:
            combined[var] = val
    return combined

def unify(e1, e2, substitution=None):
    if substitution is None:
        substitution = {}
    e1_sub = apply_substitution(e1, substitution)
    e2_sub = apply_substitution(e2, substitution)
    if e1_sub == e2_sub:
        return substitution
    if is_variable(e1_sub):
        return unify_variable(e1_sub, e2_sub, substitution)
    if is_variable(e2_sub):
        return unify_variable(e2_sub, e1_sub, substitution)
    if isinstance(e1_sub, list) and isinstance(e2_sub, list):
        if len(e1_sub) != len(e2_sub) or e1_sub[0] != e2_sub[0]:
            return "FAILURE"
        for a, b in zip(e1_sub[1:], e2_sub[1:]):
            a_sub = apply_substitution(a, substitution)
            b_sub = apply_substitution(b, substitution)
            theta = unify(a_sub, b_sub, substitution)
            if theta == "FAILURE":
                return "FAILURE"
            substitution = combine(substitution, theta)
        return substitution
    return "FAILURE"

def unify_variable(v, e, substitution):
    if v == e:
        return substitution
    if v in substitution:
        return unify(substitution[v], e, substitution)
    e_sub = apply_substitution(e, substitution)
    if occurs_in(v, e_sub, substitution):
        return "FAILURE"
    new_sub = {v: e_sub}
    return combine(substitution, new_sub)

def format_expr(expr):
    if isinstance(expr, list):
        return f"{expr[0]}({', '.join(format_expr(arg) for arg in expr[1:])})"
    return str(expr)

In [16]:
E1_input = "P(x)"
E2_input = "P(Ram)"

parsed_e1 = parse_expression(E1_input)
parsed_e2 = parse_expression(E2_input)

print(f"Expression 1: {E1_input}")
print(f"Expression 2: {E2_input}")

result = unify(parsed_e1, parsed_e2)
if result == "FAILURE":
    print("\nResult: FAILURE")
else:
    formatted_subs = [f"{key}/{format_expr(val)}" for key, val in result.items()]
    print(f"\nResult: {{{', '.join(formatted_subs)}}}")

Expression 1: P(x)
Expression 2: P(Ram)

Result: {x/Ram}
